# Wallet Strategy Selection

Unified selection stage: runs **all wallet-selection methods** and saves each
as a named `WalletSelectionStrategy` artifact in the workspace.  The backtest
stage loads these artifacts directly — no strategy logic is re-built there.

## Methods included
| id suffix | description |
|-----------|-------------|
| `skill_sweep` → `quality_core` | top-N by best skill metric (grid-searched on train-a→train-b) |
| `cohort_early_entry` | wallets that enter markets early |
| `cohort_smooth_pnl` | wallets with high PnL relative to volatility |
| `volatility` | volatility-filtered wallets (from the profitable_wallet_analysis path) |

Each method produces **two trigger variants**:
- `score_threshold` — composite signal score ≥ calibrated threshold (Kelly sizing)
- `all_open_buys`   — every open-buy event (fixed stake baseline)


In [525]:

%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [526]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [527]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-02-15  TRAIN_A_END_DATE=2026-01-27


## Compute / load wallet skill metrics

In [533]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,925
train_b_wallets,925
full_train_wallets,925
test_wallets,925


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [534]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,340,0.080833,0.060103,0.319926,18043.921825,0.447059
20,roi_sharpe,50,50,315,0.211007,0.042803,0.267965,168591.010789,0.647619
11,avg_copy_roi_capped,100,100,711,0.218082,0.064467,0.226451,296936.492582,0.503516
21,roi_sharpe,100,100,676,0.204206,0.052322,0.205070,295374.414162,0.600592
15,edge_sharpe,50,50,429,0.170358,0.032306,0.204356,164824.032099,0.650350
16,edge_sharpe,100,100,860,0.184495,0.042264,0.152112,310615.583498,0.594186
0,prob_edge_shrunk,50,50,634,0.188443,0.040407,0.132903,252486.756567,0.504732
1,prob_edge_shrunk,100,100,1756,0.098239,0.036144,0.126535,274326.446208,0.507973
30,hit_rate,50,50,649,0.104465,0.020843,0.120614,160507.064978,0.724191
31,hit_rate,100,100,890,0.094608,0.029219,0.109206,301699.135186,0.685393


In [535]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Select wallets (skill path) + build cohorts

In [536]:
selected_wallets = select_wallets(full_train_metrics, BEST_SELECTION_METRIC, BEST_TOP_N)
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)
selected_wallets[[
    "wallet", "open_buy_trades", "distinct_markets",
    "recent_open_buy_trades", BEST_SELECTION_METRIC, "wallet_quality"
]].head(15)


,wallet,open_buy_trades,distinct_markets,recent_open_buy_trades,avg_copy_roi_capped,wallet_quality
0,0x5ad5c4608c4661361b91c92e1091d2c5b43c37b9,34,30,34,4.321652,1.00
1,0xbb0bd109b9f0c2a59b8819c466f064cf65ab3790,51,48,51,3.849986,0.98
2,0x83c821d1be2143f5699a07310101a52140c826f0,112,99,112,0.787277,0.96
3,0xf5d4c1341267162afc3749908631087db95aa077,23,23,23,0.723031,0.94
4,0x7d5c8d502cdc2084d523d0138c1a2ac3165b0182,21,21,21,0.591030,0.92
5,0xeb22f4735df6bd5d14e1b5fd10faefa7757d808a,22,22,21,0.577857,0.90
6,0x3f085518960a0462b5b9dc076c909dd77cd7ed44,59,42,59,0.528484,0.88
7,0x3a7afc4d74256cd7324ec46c7a92b93e5b3415a4,27,26,27,0.526823,0.86
8,0x03f96dca12672fd51cc0300772e379f86c64f78e,20,18,20,0.517167,0.84
9,0x45d59c4d70128ce7969124b515549aafb90e47ef,21,20,21,0.500501,0.82


## Build wallet profiles and signal events

In [537]:
from polymarket_analysis.signal.builder import (
    build_wallet_profiles,
    build_signal_events,
)

selected_wallet_profiles = build_wallet_profiles(
    dataset, selected_wallets, period="full_train",
    end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
)
selected_wallet_profiles.to_parquet(
    WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
)

if CALIBRATION_SIGNALS_PATH.exists():
    train_b_open_buys = pd.read_parquet(CALIBRATION_SIGNALS_PATH)
else:
    _, train_b_open_buys = build_signal_events(
        dataset, selected_wallet_profiles, period="train_b",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    train_b_open_buys.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

if TEST_SIGNALS_PATH.exists():
    test_open_buys = pd.read_parquet(TEST_SIGNALS_PATH)
else:
    _, test_open_buys = build_signal_events(
        dataset, selected_wallet_profiles, period="test",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    test_open_buys.to_parquet(TEST_SIGNALS_PATH, index=False)

print(f"train_b signals: {len(train_b_open_buys):,}  test signals: {len(test_open_buys):,}")


train_b signals: 823  test signals: 1,061


## Calibrate signal scoring on train-B

In [538]:
from polymarket_analysis.signal.scorer import (
    build_calibration_tables,
    apply_signal_score,
    select_signal_threshold,
)

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
print(f"Global edge: {global_edge:.4f}")
print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# score distribution
score_deciles = (
    train_b_scored.assign(
        score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
    )
    .groupby("score_decile")
    .agg(
        count=("signal_score", "size"),
        avg_signal_score=("signal_score", "mean"),
        avg_copy_roi_capped=("copy_roi_capped", "mean"),
    )
    .reset_index()
)
score_deciles


Global edge: 0.1409
Selected signal threshold: 0.85


,score_decile,count,avg_signal_score,avg_copy_roi_capped
0,0,83,0.266107,0.051682
1,1,82,0.356259,0.199901
2,2,83,0.402706,-0.007082
3,3,81,0.447969,0.063176
4,4,83,0.483517,0.313259
5,5,82,0.535564,0.380625
6,6,83,0.599245,0.869033
7,7,81,0.674816,0.445006
8,8,82,0.747892,1.704406
9,9,83,0.860832,2.376655


## Build wallet cohorts (skill path)

In [539]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

wallet_cohorts = build_wallet_cohorts(
    full_train_metrics, train_b_open_buys, selected_wallets,
    top_n=BEST_TOP_N,
)
{name: len(df) for name, df in wallet_cohorts.items()}


{'quality_core': 50, 'early_entry': 47, 'smooth_pnl': 50}

## Volatility-based wallet selection (second method)

Loads the full training set, applies the volatility filter, and stores the
result as an additional `WalletSelectionStrategy`.  The volatility wallet set
is added to `wallet_cohorts` so the strategy factory handles it uniformly.

In [540]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

# Load full training trades for the volatility path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_train_vol = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)
df_train_vol["dt"] = pd.to_datetime(df_train_vol["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_train_vol.columns and "quantity" not in df_train_vol.columns:
    df_train_vol = df_train_vol.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_train_vol["usdc_amount"]      = df_train_vol["usdc_amount"].astype(float)
df_train_vol["final_value_usdc"] = df_train_vol["final_value_usdc"].astype(float)
df_train_vol["quantity"]         = df_train_vol["quantity"].astype(float)

# PnL and notional per fill
df_train_vol["pnl"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["final_value_usdc"] - df_train_vol["usdc_amount"],
    df_train_vol["usdc_amount"] - df_train_vol["final_value_usdc"],
)
df_train_vol["notional"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["usdc_amount"],
    df_train_vol["quantity"] * (1 - df_train_vol["price"].astype(float)),
)
df_slice = df_train_vol[df_train_vol["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
del df_train_vol, df_slice

In [541]:
filtered_wallets_vol = filter_wallets_by_volatility(
    wallet_vol_train,
    min_buckets=20,
    max_top5_pnl_pct=1,
    max_top_market_pnl_pct=0.5,
)

filtered_wallets_vol = filtered_wallets_vol[filtered_wallets_vol['average_roi'] >= 0.04].sort_values("pnl_volatility", ascending=True).head(100)

print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# Build wallet_quality based on total_pnl rank (higher = better)
filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
    method="first", pct=True
)

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
vol_wallets_with_signals = set(train_b_open_buys["wallet"]).intersection(
    set(filtered_wallets_vol["wallet"])
)
wallet_cohorts["volatility"] = filtered_wallets_vol.copy().reset_index(drop=True)

# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

print(f"Volatility cohort: {len(wallet_cohorts['volatility'])}")

Volatility-filtered wallets: 100
Volatility cohort: 100


## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [542]:
from polymarket_analysis.wallet_selection.selector import build_strategies_from_sweep
from polymarket_analysis.strategy.registry import save_strategy

all_strategies = build_strategies_from_sweep(
    wallet_cohorts=wallet_cohorts,
    signal_threshold=SIGNAL_THRESHOLD,
    selection_metric=BEST_SELECTION_METRIC,
    top_n=BEST_TOP_N,
    sweep_df=cohort_sweep,
    extra_metadata={
        "end_date_train": str(END_DATE_TRAIN),
        "train_a_end_date": str(TRAIN_A_END_DATE),
    },
)

for strategy in all_strategies:
    parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
    print(f"Saved [{strategy.strategy_id}]  wallets={len(strategy.wallets):3d}  trigger={strategy.trigger.fn_ref.split('.')[-1]}")

print(f"\nTotal strategies saved: {len(all_strategies)}")


Saved [quality_core__score_threshold]  wallets= 50  trigger=score_threshold
Saved [quality_core__all_open_buys]  wallets= 50  trigger=all_open_buys
Saved [early_entry__score_threshold]  wallets= 47  trigger=score_threshold
Saved [early_entry__all_open_buys]  wallets= 47  trigger=all_open_buys
Saved [smooth_pnl__score_threshold]  wallets= 50  trigger=score_threshold
Saved [smooth_pnl__all_open_buys]  wallets= 50  trigger=all_open_buys
Saved [volatility__score_threshold]  wallets=100  trigger=score_threshold
Saved [volatility__all_open_buys]  wallets=100  trigger=all_open_buys

Total strategies saved: 8


## Strategy registry summary

In [543]:
from polymarket_analysis.strategy.registry import load_all_strategies

registry = load_all_strategies(WORKSPACE_DIR)
summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        "strategy_id": s.strategy_id,
        "name": s.name,
        "selection_method": s.selection_method,
        "num_wallets": len(s.wallets),
        "trigger_fn": s.trigger.fn_ref.split(".")[-1],
        "threshold": s.trigger.params.get("threshold"),
        "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
    })

pd.DataFrame(summary_rows)


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,early_entry__all_open_buys,early_entry | all open-buys (fixed stake),cohort_early_entry,47,all_open_buys,NaN,False
1,early_entry__score_threshold,early_entry | score >= 0.85 (Kelly),cohort_early_entry,47,score_threshold,0.85,True
2,quality_core__all_open_buys,quality_core | all open-buys (fixed stake),skill_sweep,50,all_open_buys,NaN,False
3,quality_core__score_threshold,quality_core | score >= 0.85 (Kelly),skill_sweep,50,score_threshold,0.85,True
4,smooth_pnl__all_open_buys,smooth_pnl | all open-buys (fixed stake),cohort_smooth_pnl,50,all_open_buys,NaN,False
5,smooth_pnl__score_threshold,smooth_pnl | score >= 0.85 (Kelly),cohort_smooth_pnl,50,score_threshold,0.85,True
6,volatility__all_open_buys,volatility | all open-buys (fixed stake),volatility,100,all_open_buys,NaN,False
7,volatility__score_threshold,volatility | score >= 0.85 (Kelly),volatility,100,score_threshold,0.85,True


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [544]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [545]:
_trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

In [546]:
len(wallet_cohorts['volatility'])

100

In [547]:
vol_fills = df_fills[
    (df_fills["wallet"].isin(wallet_cohorts["volatility"]["wallet"]))
    & (df_fills["is_train"] == False)
].sort_values("dt")
len(vol_fills)

418076

In [549]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    _trade_files = sorted(TRADES_DIR.glob("*.parquet"))
    df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
    df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)

    # Normalise grouped schema → canonical fill-level names
    if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
        df_fills = df_fills.rename(columns={
            "total_quantity": "quantity",
            "avg_price": "price",
            "trade_value_usdc": "usdc_amount",
        })

    df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
    df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
    df_fills["quantity"]         = df_fills["quantity"].astype(float)

    split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

    del df_fills
else:
    print("PLOT_WALLET_PNL=False — skipping plots")
